<a href="https://colab.research.google.com/github/VitorrSan/recomendador-livros/blob/main/Limpeza_de_dado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn sentence-transformers

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

BASE = "/content/drive/MyDrive/Recomendador de Livros"

DATASETS = f"{BASE}/datasets"
RESULTADOS = f"{BASE}/resultados"
MODELOS = f"{BASE}/modelos"

os.makedirs(DATASETS, exist_ok=True)
os.makedirs(RESULTADOS, exist_ok=True)
os.makedirs(MODELOS, exist_ok=True)

print("Ambiente configurado!")

Ambiente configurado!


In [ ]:
import pandas as pd

arquivo = f"{DATASETS}/goodreads_best_books_2020_2025.csv"

df = pd.read_csv(arquivo)

display(df.head())

,isbn,asin,title,book_cover,author,avg_rating,num_ratings,num_reviews,year,genre,pages,publisher,language,description,url,scraped_at
0,NaN,B07QPHT8CB,The House in the Cerulean Sea,https://m.media-amazon.com/images/S/compressed...,T.J. Klune,4.36,1020173.0,152296.0,2020.0,Fantasy,393.0,Tor Books,English,A magical island. A dangerous task. A burning ...,https://www.goodreads.com/book/show/45047384-t...,2026-04-06T11:43:18Z
1,9.781636e+12,1635574048,House of Earth and Blood,https://m.media-amazon.com/images/S/compressed...,Sarah J. Maas,4.45,1183081.0,114958.0,2020.0,Fantasy,803.0,Bloomsbury Publishing,English,Bryce Quinlan had the perfect life-working har...,https://www.goodreads.com/book/show/44778083-h...,2026-04-06T11:43:25Z
2,9.780765e+12,0765387565,The Invisible Life of Addie LaRue,https://m.media-amazon.com/images/S/compressed...,V.E. Schwab,4.16,1551956.0,196463.0,2020.0,Fantasy,448.0,Tor Books,English,A Life No One Will Remember. A Story You Will ...,https://www.goodreads.com/book/show/50623864-t...,2026-04-06T11:43:30Z
3,9.780526e+12,0525559477,The Midnight Library,https://m.media-amazon.com/images/S/compressed...,Matt Haig,3.98,2528798.0,252564.0,2020.0,Fiction,288.0,Viking,English,Between life and death there is a library.When...,https://www.goodreads.com/book/show/52578297-t...,2026-04-06T11:43:36Z
4,NaN,NaN,American Dirt,https://m.media-amazon.com/images/S/compressed...,Jeanine Cummins,4.37,700358.0,53355.0,2020.0,Fiction,459.0,Flatiron Books,English,"Lydia lives in Acapulco. She has a son, Luca, ...",https://www.goodreads.com/book/show/45046527-a...,2026-04-06T11:43:44Z


In [ ]:

print("Quantidade de livros:", len(df))

print("\nColunas:")
print(df.columns.tolist())

print("\nInformações:")
df.info()

print("\nDados ausentes:")
print(df.isnull().sum())

Quantidade de livros: 1799

Colunas:
['isbn', 'asin', 'title', 'book_cover', 'author', 'avg_rating', 'num_ratings', 'num_reviews', 'year', 'genre', 'pages', 'publisher', 'language', 'description', 'url', 'scraped_at']

Informações:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1799 entries, 0 to 1798
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   isbn         1515 non-null   float64
 1   asin         1724 non-null   object 
 2   title        1799 non-null   object 
 3   book_cover   1796 non-null   object 
 4   author       1796 non-null   object 
 5   avg_rating   1796 non-null   float64
 6   num_ratings  1796 non-null   float64
 7   num_reviews  1796 non-null   float64
 8   year         1796 non-null   float64
 9   genre        1795 non-null   object 
 10  pages        1785 non-null   float64
 11  publisher    1724 non-null   object 
 12  language     1799 non-null   object 
 13  description  1796 non-null

In [ ]:
import pandas as pd

df = pd.read_csv(arquivo)

traducao_colunas = {
    "title": "titulo",
    "book_cover": "capa_do_livro",
    "author": "autor",
    "avg_rating": "media_avaliacao",
    "num_ratings": "num_avaliacao",
    "num_reviews": "num_reviews",
    "year": "ano_publicacao",
    "genre": "genero",
    "publisher": "editora",
    "language": "idioma",
    "description": "sinopse",
    "url": "url",
    "scraped_at": "scraped_at"
}

df = df.rename(columns=traducao_colunas)

print(df.columns.tolist())

['isbn', 'asin', 'titulo', 'capa_do_livro', 'autor', 'media_avaliacao', 'num_avaliacao', 'num_reviews', 'ano_publicacao', 'genero', 'pages', 'editora', 'idioma', 'sinopse', 'url', 'scraped_at']


In [ ]:
!pip install -q transformers==4.46.3 sentencepiece sacremoses

In [ ]:
import transformers

print("Versão ativa:", transformers.__version__)

Versão ativa: 5.22.1


In [ ]:
import transformers
import sys

print("Versão:", transformers.__version__)
print("Localização:", transformers.__file__)
print("Python:", sys.version)

Versão: 5.22.1
Localização: /usr/local/lib/python3.13/dist-packages/transformers/__init__.py
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nome_modelo = "Helsinki-NLP/opus-mt-en-ROMANCE"

# Carregar o tokenizador
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)

# Carregar o modelo
modelo = AutoModelForSeq2SeqLM.from_pretrained(nome_modelo)

# Identificar o dispositivo disponível
dispositivo = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

modelo.to(dispositivo)

print("Modelo carregado!")
print("Dispositivo:", dispositivo)

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [ ]:

def traduzir_texto(texto):

    if texto is None or not str(texto).strip():
        return ""

    texto = ">>pt_BR<< " + str(texto)

    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    entradas = {
        chave: valor.to(dispositivo)
        for chave, valor in entradas.items()
    }

    with torch.no_grad():
        resultado = modelo.generate(
            **entradas,
            max_new_tokens=512
        )

    traducao = tokenizer.decode(
        resultado[0],
        skip_special_tokens=True
    )

    return traducao

In [ ]:
texto = "A young wizard discovers a magical world."

traducao = traduzir_texto(texto)

print("Original:", texto)
print("Tradução:", traducao)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original: A young wizard discovers a magical world.
Tradução: Um jovem assistente descobre um mundo mágico.


In [ ]:
print("Dimensões do dataset:", df.shape)

print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nPrimeiros 5 livros:")
display(df.head())

print("\nInformações dos dados:")
df.info()

print("\nValores ausentes:")
print(df.isnull().sum())

Dimensões do dataset: (1799, 16)

Colunas disponíveis:
['isbn', 'asin', 'title', 'book_cover', 'author', 'avg_rating', 'num_ratings', 'num_reviews', 'year', 'genre', 'pages', 'publisher', 'language', 'description', 'url', 'scraped_at']

Primeiros 5 livros:


,isbn,asin,title,book_cover,author,avg_rating,num_ratings,num_reviews,year,genre,pages,publisher,language,description,url,scraped_at
0,NaN,B07QPHT8CB,The House in the Cerulean Sea,https://m.media-amazon.com/images/S/compressed...,T.J. Klune,4.36,1020173.0,152296.0,2020.0,Fantasy,393.0,Tor Books,English,A magical island. A dangerous task. A burning ...,https://www.goodreads.com/book/show/45047384-t...,2026-04-06T11:43:18Z
1,9.781636e+12,1635574048,House of Earth and Blood,https://m.media-amazon.com/images/S/compressed...,Sarah J. Maas,4.45,1183081.0,114958.0,2020.0,Fantasy,803.0,Bloomsbury Publishing,English,Bryce Quinlan had the perfect life-working har...,https://www.goodreads.com/book/show/44778083-h...,2026-04-06T11:43:25Z
2,9.780765e+12,0765387565,The Invisible Life of Addie LaRue,https://m.media-amazon.com/images/S/compressed...,V.E. Schwab,4.16,1551956.0,196463.0,2020.0,Fantasy,448.0,Tor Books,English,A Life No One Will Remember. A Story You Will ...,https://www.goodreads.com/book/show/50623864-t...,2026-04-06T11:43:30Z
3,9.780526e+12,0525559477,The Midnight Library,https://m.media-amazon.com/images/S/compressed...,Matt Haig,3.98,2528798.0,252564.0,2020.0,Fiction,288.0,Viking,English,Between life and death there is a library.When...,https://www.goodreads.com/book/show/52578297-t...,2026-04-06T11:43:36Z
4,NaN,NaN,American Dirt,https://m.media-amazon.com/images/S/compressed...,Jeanine Cummins,4.37,700358.0,53355.0,2020.0,Fiction,459.0,Flatiron Books,English,"Lydia lives in Acapulco. She has a son, Luca, ...",https://www.goodreads.com/book/show/45046527-a...,2026-04-06T11:43:44Z



Informações dos dados:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1799 entries, 0 to 1798
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   isbn         1515 non-null   float64
 1   asin         1724 non-null   object 
 2   title        1799 non-null   object 
 3   book_cover   1796 non-null   object 
 4   author       1796 non-null   object 
 5   avg_rating   1796 non-null   float64
 6   num_ratings  1796 non-null   float64
 7   num_reviews  1796 non-null   float64
 8   year         1796 non-null   float64
 9   genre        1795 non-null   object 
 10  pages        1785 non-null   float64
 11  publisher    1724 non-null   object 
 12  language     1799 non-null   object 
 13  description  1796 non-null   object 
 14  url          1799 non-null   object 
 15  scraped_at   1799 non-null   object 
dtypes: float64(6), object(10)
memory usage: 225.0+ KB

Valores ausentes:
isbn           284
asin            75
t

In [ ]:
df_limpo = df.copy()

colunas_texto = [
    "title",
    "author",
    "genre",
    "publisher",
    "language",
    "description"
]

for coluna in colunas_texto:
    df_limpo[coluna] = (
        df_limpo[coluna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

print("Preparação inicial concluída!")

Preparação inicial concluída!


In [ ]:
duplicados = df_limpo[
    df_limpo.duplicated(
        subset=["title", "author"],
        keep=False
    )
].sort_values(["title", "author"])

print("Quantidade de registros duplicados:",
      len(duplicados))

display(
    duplicados[
        ["title", "author", "genre", "year"]
    ]
)

Quantidade de registros duplicados: 5


,title,author,genre,year
1133,"Sorry, we couldn’t find the page you were look...",<NA>,<NA>,NaN
1145,"Sorry, we couldn’t find the page you were look...",<NA>,<NA>,NaN
1725,"Sorry, we couldn’t find the page you were look...",<NA>,<NA>,NaN
1629,The Loneliness of Sonia and Sunny,Kiran Desai,Fiction,2025.0
1637,The Loneliness of Sonia and Sunny,Kiran Desai,Fiction,2025.0


In [ ]:
print(
    df_limpo.isnull().sum()
    .sort_values(ascending=False)
)

isbn           284
asin            75
publisher       75
pages           14
genre            4
num_ratings      3
book_cover       3
author           3
description      3
num_reviews      3
avg_rating       3
year             3
title            0
language         0
url              0
scraped_at       0
dtype: int64


In [ ]:
# 1. Remover registros que são mensagens de erro
erros = df_limpo["title"].str.contains(
    "Sorry, we couldn",
    case=False,
    na=False
)

df_limpo = df_limpo.loc[~erros].copy()

# 2. Remover livros duplicados, mantendo a primeira ocorrência
df_limpo = df_limpo.drop_duplicates(
    subset=["title", "author"],
    keep="first"
).copy()

# 3. Conferir o resultado
print("Registros restantes:", len(df_limpo))
print("\nDuplicatas restantes:")
print(df_limpo.duplicated(subset=["title", "author"]).sum())

print("\nDados ausentes:")
print(df_limpo.isnull().sum())

Registros restantes: 1795

Duplicatas restantes:
0

Dados ausentes:
isbn           281
asin            72
title            0
book_cover       0
author           0
avg_rating       0
num_ratings      0
num_reviews      0
year             0
genre            1
pages           11
publisher       72
language         0
description      0
url              0
scraped_at       0
dtype: int64


In [ ]:
caminho_limpo = "/content/drive/MyDrive/Recomendador de Livros/datasets/livros_limpos.csv"

df_limpo.to_csv(caminho_limpo, index=False, encoding="utf-8-sig")

print("Dataset limpo salvo com sucesso!")

Dataset limpo salvo com sucesso!


In [ ]:


# Criar uma cópia para preservar o dataset original
df_modelo = df_limpo.copy()

# Substituir gêneros ausentes por texto vazio
df_modelo["genre"] = df_modelo["genre"].fillna("")

# Combinar as informações relevantes de cada livro
df_modelo["texto_combinado"] = (
    "Título: " + df_modelo["title"].astype(str) + ". " +
    "Autor: " + df_modelo["author"].astype(str) + ". " +
    "Gênero: " + df_modelo["genre"].astype(str) + ". " +
    "Descrição: " + df_modelo["description"].astype(str)
)

# Conferir os resultados
print("Quantidade de textos:", len(df_modelo))
print("\nExemplo de texto combinado:\n")
print(df_modelo["texto_combinado"].iloc[0])

Quantidade de textos: 1795

Exemplo de texto combinado:

Título: The House in the Cerulean Sea. Autor: T.J. Klune. Gênero: Fantasy. Descrição: A magical island. A dangerous task. A burning secret.Linus Baker leads a quiet, solitary life. At forty, he lives in a tiny house with a devious cat and his old records. As a Case Worker at the Department in Charge Of Magical Youth, he spends his days overseeing the well-being of children in government-sanctioned orphanages.When Linus is unexpectedly summoned by Extremely Upper Management he's given a curious and highly classified assignment: travel to Marsyas Island Orphanage, where six dangerous children reside: a gnome, a sprite, a wyvern, an unidentifiable green blob, a were-Pomeranian, and the Antichrist. Linus must set aside his fears and determine whether or not they’re likely to brin
